# RQ2 — Generalization: Does LLM Guidance Scale Across Scenarios?

**Research question:** Does LLM-guided RL maintain its performance advantage as network
complexity grows? We compare methods across four NASimLLM network scenarios of increasing
difficulty.

**Scenarios (ordered by complexity)**

| Order | Scenario | Hosts | Topology | Notes |
|---|---|---|---|---|
| 1 | `tiny` | 3 | star | smallest network |
| 2 | `small` | ~8 | tree | moderately connected |
| 3 | `small-linear` | ~8 | linear chain | **same host count as `small`**, different topology — annotated distinctly |
| 4 | `medium` | ~16 | mixed | largest network tested |

**Data layout:** `runs/{run_tag}/rq2/{scenario}/{condition}/seed{N}/train.csv`
(the loader defaults to the `llm_full` condition; this matches the output of
`nasim.scripts.run_rq2` and `hpc/launch.sh`).

**Performance metric:** Success rate (fraction of episodes where all targets were compromised)
over the final 10 training episodes. Success rate is topology-agnostic — it maps cleanly to
[0, 1] regardless of reward scale differences between scenarios, making cross-scenario
comparison honest without ad-hoc normalisation.

In [ ]:
"""Imports and configuration — RQ2."""
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from pathlib import Path
from scipy import stats

# ── Output directory ──────────────────────────────────────────────────────────
# ── Resolve project root robustly ────────────────────────────────────
import os as _os
try:
    # VS Code injects __vsc_ipynb_file__ — most reliable
    _project_root = Path(__vsc_ipynb_file__).resolve().parent.parent
except NameError:
    # Fallback: walk up from CWD to find the nasim package marker
    _cwd = Path(_os.getcwd()).resolve()
    for _p in [_cwd] + list(_cwd.parents):
        if (_p / "nasim").is_dir():
            _project_root = _p
            break
    else:
        _project_root = _cwd

# -- Output directory --
FIGURES_DIR = _project_root / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

RUNS_ROOT = _project_root / "runs"

# ── Scenario ordering (x-axis) ────────────────────────────────────────────────
SCENARIOS = ["tiny", "small", "small-linear", "medium"]
SCENARIO_LABELS = {
    "tiny":         "Tiny\n(3 hosts)",
    "small":        "Small\n(~8 hosts)",
    "small-linear": "Small-Linear\n(~8 hosts, chain)",
    "medium":       "Medium\n(~16 hosts)",
}
TOPOLOGY_VARIANT = {"small-linear"}   # same host count as predecessor, different graph

SEEDS = list(range(10))

# ── Runs to compare ───────────────────────────────────────────────────────────
# Generalisation panel: one llm_full line per teacher across all scenarios, plus
# the no-LLM (ppo_options) reference from the primary teacher.
# Path expected:  runs/{model_tag}/rq2/{scenario}/{condition}/seed{N}/train.csv
#   tag       = unique display key (legend + all_data key)
#   model_tag = folder under runs/ (defaults to tag)
#   condition = llm_full (default) or ppo_options
RUNS = [
    {"tag": "qwen-4B",  "label": "Qwen-4B",  "color": "#0072B2", "ls": "-",  "marker": "o"},
    {"tag": "llama-3B", "label": "Llama-3B", "color": "#D55E00", "ls": "--", "marker": "s"},
    {"tag": "llama-8B", "label": "Llama-8B", "color": "#CC79A7", "ls": "-.", "marker": "^"},
    {"tag": "PPO (no LLM)", "model_tag": "qwen-4B", "condition": "ppo_options",
     "label": "PPO + Options (no LLM)", "color": "#999999", "ls": ":", "marker": "x"},
]

# ── Matplotlib style ──────────────────────────────────────────────────────────
plt.rcParams.update({
    "figure.dpi":      150,
    "figure.figsize":  (10, 5),
    "font.size":       12,
    "axes.titlesize":  14,
    "axes.labelsize":  13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
    "axes.grid":       True,
    "grid.alpha":      0.3,
    "lines.linewidth": 2.0,
})

def save_figure(fig, name: str):
    png = FIGURES_DIR / f"{name}.png"
    pdf = FIGURES_DIR / f"{name}.pdf"
    fig.savefig(png, dpi=300, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    print(f"  Saved: {png}")
    print(f"  Saved: {pdf}")

print("Configuration loaded.")
print(f"  Runs      : {[r['tag'] for r in RUNS]}")
print(f"  Scenarios : {SCENARIOS}")

## 1. Data Discovery & Schema Inspection

In [ ]:
def load_rq2_run(model_tag: str, scenarios: list, seeds: list, condition: str = "llm_full") -> dict:
    """Return {scenario: {seed: DataFrame}} for one model_tag × condition.

    Path: runs/{model_tag}/rq2/{scenario}/{condition}/seed{N}/train.csv
    """
    base   = RUNS_ROOT / model_tag / "rq2"
    result = {}
    for scenario in scenarios:
        result[scenario] = {}
        for seed in seeds:
            csv = base / scenario / condition / f"seed{seed}" / "train.csv"
            if csv.exists():
                df = pd.read_csv(csv)
                if len(df) > 0:
                    df["seed"]      = seed
                    df["scenario"]  = scenario
                    df["run"]       = model_tag
                    df["condition"] = condition
                    result[scenario][seed] = df
                else:
                    print(f"  [WARN] {csv.relative_to(RUNS_ROOT)} — 0 rows, skipping")
    return result


# all_data[run["tag"]][scenario] = {seed: DataFrame}
# run["tag"] is unique per entry; model_tag is the filesystem path.
all_data = {}
for run in RUNS:
    model_tag = run.get("model_tag", run["tag"])
    condition = run.get("condition", "llm_full")
    all_data[run["tag"]] = load_rq2_run(model_tag, SCENARIOS, SEEDS, condition=condition)

# ── Schema inspection ──────────────────────────────────────────────────────────
print("=" * 70)
print("SCHEMA INSPECTION — train.csv (RQ2)")
print("=" * 70)
sample = None
for run in RUNS:
    for scen in SCENARIOS:
        sd = all_data[run["tag"]].get(scen, {})
        if sd:
            sample = next(iter(sd.values()))
            print(f"\nSample: run={run['tag']}  scenario={scen}  seed={next(iter(sd))}")
            break
    if sample is not None:
        break

if sample is not None:
    print(f"Shape  : {sample.shape}")
    print("\nColumn dtypes:")
    print(sample.dtypes.to_string())
    print("\nFirst 3 rows:")
    print(sample.drop(columns=["seed","scenario","run","condition"], errors="ignore").head(3).to_string())

# ── Availability table ─────────────────────────────────────────────────────────
print("\n" + "=" * 70)
print("AVAILABILITY SUMMARY (RQ2)")
print("=" * 70)
for run in RUNS:
    for scen in SCENARIOS:
        sd          = all_data[run["tag"]].get(scen, {})
        seeds_found = sorted(sd.keys())
        ep_counts   = [len(sd[s]) for s in seeds_found]
        print(f"  {run['tag']:35s}  {scen:15s}  seeds={seeds_found}  ep={ep_counts}")
print("=" * 70)

# ── Teacher coverage check ─────────────────────────────────────────────────────
if len(RUNS) > 1:
    print("\n" + "=" * 70)
    print("TEACHER COVERAGE CHECK")
    print("=" * 70)
    coverage = {run["tag"]: {scen for scen in SCENARIOS
                              if all_data[run["tag"]].get(scen)}
                for run in RUNS}
    all_equal = len(set(frozenset(v) for v in coverage.values())) == 1
    if all_equal:
        print("  All runs cover the same scenarios — safe to overlay on the same axes. ✓")
    else:
        for tag, covered in coverage.items():
            missing = set(SCENARIOS) - covered
            if missing:
                print(f"  [WARN] Run '{tag}' is MISSING scenarios: {sorted(missing)}")
                print(f"         Its slope line will have gaps for those scenarios.")
    print("=" * 70)
else:
    print("\n  (Only one run configured — no coverage comparison needed.)")


### Column Mapping Notes (RQ2)

| Notebook concept | Actual column | Notes |
|---|---|---|
| Performance metric | `success` | Binary 0/1 per episode; averaged over last 10 episodes → success rate ∈ [0, 1]. Chosen over `reward` because reward scales differ across scenarios. |
| Episode index | `episode` | 1-based integer |
| Scenario identifier | folder name | Encoded in the path, not a column in `train.csv`; added as `scenario` column after loading. |

> **Normalisation choice:** Success rate (mean of `success` over last 10 episodes) is used as
> the performance metric. This is topology-agnostic and bounded in [0, 1], so no reward-scale
> normalisation is needed. Reward-based normalisation (e.g., fraction of bruteforce reward) is
> not feasible without bruteforce reference data for each scenario.

In [ ]:
"""Compute final success-rate stats per (run, scenario)."""

FINAL_WINDOW = 10

def final_perf(seed_dict: dict, metric: str, window: int):
    """Return (mean, ci_half) over the last `window` episodes, averaged across seeds."""
    per_seed = [df[metric].tail(window).mean() for df in seed_dict.values()]
    if not per_seed:
        return np.nan, np.nan
    if len(per_seed) == 1:
        return per_seed[0], 0.0
    mu = np.mean(per_seed)
    se = stats.sem(per_seed)
    ci = se * stats.t.ppf(0.975, len(per_seed) - 1)
    return mu, ci


rows = []
for run in RUNS:
    for scenario in SCENARIOS:
        sd     = all_data[run["tag"]].get(scenario, {})
        mu, ci = final_perf(sd, "success", FINAL_WINDOW)
        rows.append({
            "run_tag":  run["tag"],
            "label":    run["label"],
            "scenario": scenario,
            "mean":     mu,
            "ci":       ci,
            "n_seeds":  len(sd),
        })

perf_df = pd.DataFrame(rows)

print("Final success rate (last 10 episodes, per run × scenario):")
print(perf_df.to_string(index=False))

## 2. Figure 1 — Generalisation Slope Plot

**What this shows:** Final task-success rate (mean over the last 10 training episodes) for each
method across four scenarios. Success rate is the primary metric here — it is bounded [0, 1]
and free of the reward-scale differences that arise from different network sizes and KL penalties,
so cross-scenario comparison is fair without additional normalisation.

**Reading the x-axis carefully:** `small-linear` sits between `small` and `medium` on the axis
but is **not a size step** — it has the same number of hosts as `small` with a different
topology (linear chain rather than tree), which forces the agent to pivot through hosts
sequentially and tends to be harder. A diamond (◆) marker and a dashed vertical divider make
this explicit. Do not interpret a performance change between `small` and `small-linear` as
evidence of scaling behaviour; it is a topology effect. Scale-related conclusions should focus
on the `tiny → small → medium` progression.

In [ ]:
"""Figure 1 — Slope plot: final success rate across scenarios, one line per run."""

x_pos   = np.arange(len(SCENARIOS))
x_ticks = [SCENARIO_LABELS[s] for s in SCENARIOS]
MARKER_TOPOLOGY = "D"   # diamond for topology-variant points

fig, ax = plt.subplots(figsize=(11, 5.5))
any_plotted      = False
single_seed_note = False

for run in RUNS:
    sub = perf_df[perf_df["run_tag"] == run["tag"]]
    if sub["mean"].isna().all():
        continue

    means = sub.set_index("scenario").reindex(SCENARIOS)["mean"].values
    cis   = sub.set_index("scenario").reindex(SCENARIOS)["ci"].values
    if sub["n_seeds"].max() <= 1:
        single_seed_note = True

    for i, scen in enumerate(SCENARIOS):
        if np.isnan(means[i]):
            continue
        mkr = MARKER_TOPOLOGY if scen in TOPOLOGY_VARIANT else run["marker"]
        ax.plot(x_pos[i], means[i] * 100,
                marker=mkr, markersize=11, color=run["color"], linestyle="none",
                markeredgecolor="black", markeredgewidth=0.6, zorder=4)
        if cis[i] > 0:
            ax.errorbar(x_pos[i], means[i] * 100, yerr=cis[i] * 100,
                        fmt="none", color=run["color"], capsize=5,
                        elinewidth=1.5, zorder=3)

    valid_mask = ~np.isnan(means)
    ax.plot(x_pos[valid_mask], means[valid_mask] * 100,
            color=run["color"], linestyle=run["ls"],
            linewidth=2.0, label=run["label"], zorder=2)
    any_plotted = True

# ── small-linear topology annotation ─────────────────────────────────────────
tv_idx = SCENARIOS.index("small-linear")
ax.axvspan(tv_idx - 0.4, tv_idx + 0.4, color="#EEEEEE", alpha=0.7, zorder=0, linewidth=0)
ax.axvline(tv_idx, color="#999999", linestyle="--", linewidth=1.0, alpha=0.9, zorder=1)
ax.annotate(
    "Topology variant\n(same hosts as Small,\nchain graph ≠ tree)",
    xy=(tv_idx, 104), xytext=(tv_idx + 0.55, 100),
    fontsize=8.5, color="#555555", ha="left", va="top",
    arrowprops=dict(arrowstyle="-", color="#999999", lw=0.8),
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white",
              edgecolor="#AAAAAA", alpha=0.9),
    zorder=5,
)

ax.set_xticks(x_pos)
ax.set_xticklabels(x_ticks, fontsize=11)
ax.set_ylabel("Final Task Success Rate (%)", fontsize=13, fontweight="bold")
ax.set_xlabel("Network Scenario", fontsize=13, fontweight="bold")
ax.set_title(
    "RQ2 — Generalisation Across Scenarios\n"
    "(success rate, last 10 episodes; ◆ = topology variant, not a size step)",
    fontsize=14, fontweight="bold", pad=12)
ax.set_ylim([-5, 115])
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=100, decimals=0))

legend_handles, _ = ax.get_legend_handles_labels()
tv_handle = plt.Line2D([0], [0], marker=MARKER_TOPOLOGY, color="#555555",
                       linestyle="none", markersize=10,
                       markeredgecolor="black", markeredgewidth=0.6,
                       label="◆ Topology variant (same host count, chain graph)")
legend_handles.append(tv_handle)
ax.legend(handles=legend_handles, loc="upper right", framealpha=0.92, fontsize=10)

if single_seed_note:
    ax.text(0.01, 0.01, "Note: 1 seed per point — error bars not available.",
            transform=ax.transAxes, fontsize=9, color="#555555", va="bottom", ha="left")
if not any_plotted:
    ax.text(0.5, 0.5, "No data found — check RUNS_ROOT.",
            transform=ax.transAxes, ha="center", va="center", fontsize=13, color="red")

plt.tight_layout()
save_figure(fig, "rq2_generalisation_slope")
plt.show()

## 3. Figure 2 — Per-Scenario Learning Curves

**What this shows:** Training success-rate curves for each scenario in a 2×2 panel. Plotting
all learning curves in one view lets the reader see whether LLM4Teach converges faster in every
scenario, not just the final score. Each subplot is self-contained with its own y-axis.

In [ ]:
"""Figure 2 — 2×2 panel: per-scenario success-rate learning curves, one line per run."""

ROLLING_W = 10

fig, axes = plt.subplots(2, 2, figsize=(13, 9), sharey=False)
axes_flat = axes.flatten()

for ax_idx, scenario in enumerate(SCENARIOS):
    ax         = axes_flat[ax_idx]
    title_str  = SCENARIO_LABELS[scenario].replace("\n", " ")
    if scenario in TOPOLOGY_VARIANT:
        title_str += "  ◆ topology variant"
    ax.set_title(title_str, fontsize=12, fontweight="bold")

    any_line       = False
    single_seed_ax = False

    for run in RUNS:
        sd = all_data[run["tag"]].get(scenario, {})
        if not sd:
            continue

        all_dfs = list(sd.values())
        min_ep  = min(len(df) for df in all_dfs)
        eps     = np.arange(1, min_ep + 1)
        mat     = np.stack([df["success"].values[:min_ep] for df in all_dfs], axis=0)

        if mat.shape[0] == 1:
            smoothed = pd.Series(mat[0]).rolling(ROLLING_W, min_periods=1, center=True).mean().values
            ax.plot(eps, smoothed * 100, color=run["color"],
                    linestyle=run["ls"], linewidth=2.0,
                    label=run["label"] + " (rolling mean)")
            single_seed_ax = True
        else:
            mu  = mat.mean(axis=0)
            se  = stats.sem(mat, axis=0)
            ci  = se * stats.t.ppf(0.975, mat.shape[0] - 1)
            ax.plot(eps, mu * 100, color=run["color"],
                    linestyle=run["ls"], linewidth=2.0, label=run["label"])
            ax.fill_between(eps, (mu - ci) * 100, (mu + ci) * 100,
                            alpha=0.18, color=run["color"])
        any_line = True

    ax.set_ylim([-5, 105])
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=100, decimals=0))
    ax.set_xlabel("Episode", fontsize=11)
    ax.set_ylabel("Success Rate (%)", fontsize=11)
    ax.legend(loc="lower right", fontsize=9)
    if single_seed_ax:
        ax.text(0.01, 0.01, "1 seed — rolling mean",
                transform=ax.transAxes, fontsize=8, color="#555555",
                va="bottom", ha="left")
    if not any_line:
        ax.text(0.5, 0.5, "No data", transform=ax.transAxes,
                ha="center", va="center", fontsize=11, color="gray")

fig.suptitle("RQ2 — Learning Curves per Scenario (success rate)",
             fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
save_figure(fig, "rq2_per_scenario_curves")
plt.show()

## 4. Export Verification

In [ ]:
expected = [
    "rq2_generalisation_slope.png",
    "rq2_generalisation_slope.pdf",
    "rq2_per_scenario_curves.png",
    "rq2_per_scenario_curves.pdf",
]
print("Export verification:")
all_ok = True
for fname in expected:
    p = FIGURES_DIR / fname
    status = "OK" if p.exists() else "MISSING"
    if status == "MISSING":
        all_ok = False
    print(f"  [{status}] {p}")
print()
if all_ok:
    print("All RQ2 figures exported successfully.")
else:
    print("Some figures are missing — re-run the plotting cells above.")